# CLINICAL TRIALS RISK ANALYSIS
---
## 01 - Data Collection
This notebook performs the first step of data pipeline: **retrieving raw clinical trial records using the modern ClinicalTrials.gov JSON API (API v2)**.

### Objective  
The objective of this step is to programmatically download the **first 5,000 clinical trial study records**, preserve their full hierarchical structure, and save them as **raw JSON files** for reproducibility.  
These unprocessed records will later be flattened, cleaned, and transformed into a tabular, ML-ready dataset in the preprocessing stage.

### Why collect raw JSON?  
ClinicalTrials.gov API v2 provides a modernized schema where each study is represented as a deeply nested JSON object containing:

- **protocolSection** $\rightarrow$ core study information  
- **identificationModule** $\rightarrow$ NCT ID, titles, organization  
- **statusModule** $\rightarrow$ recruitment and completion status, dates  
- **designModule** $\rightarrow$ study design, phases, intervention model  
- **conditionsModule** $\rightarrow$ medical conditions under study  
- **armsInterventionsModule** $\rightarrow$ arms, interventions, drug/device details  
- **outcomesModule** $\rightarrow$ primary/secondary outcomes  
- **derivedSection** $\rightarrow$ curated/standardized fields  

Working directly with the raw JSON ensures that no structural information is lost before feature extraction.

### Output  
Running this notebook will generate: `data/raw/clinical_trials_5000.json` 
containing approximately **5,000 complete study objects**, which forms the foundation for downstream preprocessing, EDA, and risk-prediction modeling.

### Imports & Config

In [9]:
import sys
import os

# Path to project root (folder that contains "src/")
project_root = os.path.abspath("..")
sys.path.append(project_root)

print("Project root added:", project_root)

Project root added: /home/thucquyen/Clinical-Trial-Failure-Prediction


In [10]:
import json
import requests

from src.data.data_loader import ClinicalTrialsAPI

BASE_URL = "https://clinicaltrials.gov/api/v2/studies"
OUTPUT_PATH = "../data/raw/clinical_trials_5000.json"

### Smoke Test: Fetch 1 Record

In [11]:
# Quick test request to check API availability & structure
response = requests.get(BASE_URL, params={"pageSize": 1})
response.raise_for_status()

sample = response.json()
sample.keys(), len(sample["studies"])

(dict_keys(['studies', 'nextPageToken']), 1)

### Inspect Structure of a Study (Schema Exploration)

In [12]:
sample_study = sample["studies"][0]

protocol = sample_study.get("protocolSection", {})

print("protocolSection keys:")
print(list(protocol.keys()))

ident = protocol.get("identificationModule", {})
status = protocol.get("statusModule", {})
design = protocol.get("designModule", {})

print("\nIdentification Module:")
print(json.dumps(ident, indent=2)[:800])

print("\nStatus Module:")
print(json.dumps(status, indent=2)[:800])

protocolSection keys:
['identificationModule', 'statusModule', 'sponsorCollaboratorsModule', 'oversightModule', 'descriptionModule', 'conditionsModule', 'designModule', 'armsInterventionsModule', 'outcomesModule', 'eligibilityModule', 'contactsLocationsModule', 'referencesModule']

Identification Module:
{
  "nctId": "NCT00660335",
  "orgStudyIdInfo": {
    "id": "RG1068-16"
  },
  "organization": {
    "fullName": "Repligen Corporation",
    "class": "INDUSTRY"
  },
  "briefTitle": "Safety and Efficacy of Synthetic Human Secretin-Enhanced MRCP in Subjects With Abnormalities of the Pancreas",
  "officialTitle": "Phase III Study to Demonstrate the Efficacy and Safety of RG1068 (Synthetic Human Secretin)- Enhanced Magnetic Resonance Cholangiopancreatography (MRCP) in the Evaluation of Subjects With a History of Acute or Acute Recurrent Pancreatitis"
}

Status Module:
{
  "statusVerifiedDate": "2009-12",
  "overallStatus": "COMPLETED",
  "expandedAccessInfo": {
    "hasExpandedAccess": fa

### Initialize API Loader (from data_loader.py)

In [13]:
api = ClinicalTrialsAPI(page_size=100, sleep=0.15, max_retries=3)

api 

### Fetch 5000 Studies

In [14]:
studies = api.fetch_n_studies(n=5000)
len(studies)

[INFO] Fetching up to 5000 studies...
[INFO] Retrieved batch with 100 studies (total: 100)
[INFO] Retrieved batch with 100 studies (total: 200)
[INFO] Retrieved batch with 100 studies (total: 300)
[INFO] Retrieved batch with 100 studies (total: 400)
[INFO] Retrieved batch with 100 studies (total: 500)
[INFO] Retrieved batch with 100 studies (total: 600)
[INFO] Retrieved batch with 100 studies (total: 700)
[INFO] Retrieved batch with 100 studies (total: 800)
[INFO] Retrieved batch with 100 studies (total: 900)
[INFO] Retrieved batch with 100 studies (total: 1000)
[INFO] Retrieved batch with 100 studies (total: 1100)
[INFO] Retrieved batch with 100 studies (total: 1200)
[INFO] Retrieved batch with 100 studies (total: 1300)
[INFO] Retrieved batch with 100 studies (total: 1400)
[INFO] Retrieved batch with 100 studies (total: 1500)
[INFO] Retrieved batch with 100 studies (total: 1600)
[INFO] Retrieved batch with 100 studies (total: 1700)
[INFO] Retrieved batch with 100 studies (total: 1800)

5000

### Quick Sanity Check

In [15]:
# Show first 10 NCT IDs
ids = [
    s.get("protocolSection", {})
     .get("identificationModule", {})
     .get("nctId")
    for s in studies[:10]
]

ids

['NCT00660335',
 'NCT04938635',
 'NCT00587535',
 'NCT00921635',
 'NCT01780935',
 'NCT07227935',
 'NCT01206335',
 'NCT01266135',
 'NCT03071835',
 'NCT01862835']

### Save Raw JSON

In [16]:
api.save_json(studies, OUTPUT_PATH)

[INFO] JSON saved to: ../data/raw/clinical_trials_5000.json


### Validate Output File

In [17]:
size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f"Saved file size: {size_mb:.2f} MB")

Saved file size: 133.12 MB
